# Random walk

Driftless Gaussian random walk in log price: `dlog p = sigma dW`. The null scenario, with nothing to find.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fantasma as fm

## Parameters

Model parameters are annualised; `dt` is the step length over a 365.25-day year.

In [ ]:
SEED = 42
N_PATHS = 5
START = "2024-01-01"
FREQ = "1d"  # Almanac frequency ("1m", "10m", "1h", "1d") or any pandas offset
N_STEPS = 365 * 5

SIGMA = 0.2
INITIAL_PRICE = 100.0
ASSET = "RW"

## Simulate

In [ ]:
model = fm.RandomWalk(sigma=SIGMA, asset=ASSET, initial_price=INITIAL_PRICE)
model

In [ ]:
paths = fm.simulate(
    model,
    start=START,
    freq=FREQ,
    n_steps=N_STEPS,
    seed=SEED,
    n_paths=N_PATHS,
    return_hidden=True,
)
prices, hidden = paths[0]
prices.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (path_prices, _) in enumerate(paths):
    for asset in path_prices.columns:
        ax.plot(path_prices.index, path_prices[asset], lw=0.8, label=f"path {i} {asset}")
ax.set_title("Random walk")
ax.set_ylabel("price")
if len(paths) * paths[0][0].shape[1] <= 10:
    ax.legend(fontsize=8)
plt.show()

## Check: realised volatility

The annualised standard deviation of log returns should be close to `SIGMA` (standard error `SIGMA / sqrt(2 n)`).

In [ ]:
dt = fm.RegularGrid(START, FREQ, n_steps=1).dt
r = np.diff(np.log(prices[ASSET].to_numpy()))
sigma_hat = np.sqrt(np.mean(r**2) / dt)
print(f"sigma = {SIGMA:.4f}, realised = {sigma_hat:.4f}, s.e. = {SIGMA / np.sqrt(2 * len(r)):.4f}")